In [20]:
%pip install pandas numpy geopy folium scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import pandas as pd
import folium
from geopy.geocoders import Nominatim, ArcGIS
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import (
    GeocoderTimedOut,
    GeocoderUnavailable,
    GeocoderServiceError,
    GeocoderRateLimited,
)
import time

# 1) GÃ©ocodeurs
osm = Nominatim(user_agent="delivery_tunis_v1")
arc = ArcGIS(timeout=10)

# Ralentisseurs
geocode_osm = RateLimiter(
    osm.geocode,
    min_delay_seconds=3.0,
    error_wait_seconds=10.0,
    max_retries=1,
    swallow_exceptions=False,
)

geocode_arc = RateLimiter(
    arc.geocode,
    min_delay_seconds=1.0,
    error_wait_seconds=5.0,
    max_retries=1,
    swallow_exceptions=False,
)

# 2) Charger le dataset
df = pd.read_csv(r"C:\Users\ghayd\Documents\p2m\V-2-V--logistics-main\central agent\Copie de client_data.csv")

print("Colonnes:", list(df.columns))
print("Nb lignes:", len(df))

_cache_gps = {}


def _try_geocode_osm(query: str):
    return geocode_osm(query, timeout=10, country_codes="tn", limit=1)


def _try_geocode_arc(query: str):
    # ArcGIS ne supporte pas country_codes comme Nominatim
    return geocode_arc(query)


def choisir_service():
    """Choisit un service de gÃ©ocodage qui rÃ©pond (OSM puis ArcGIS)."""
    test_query = "Tunis, Tunisia"

    try:
        loc = _try_geocode_osm(test_query)
        if loc:
            print("Service gÃ©ocodage: Nominatim (OSM)")
            return "osm"
    except GeocoderRateLimited:
        print("Nominatim: rate-limited (429) â€” fallback ArcGIS")
    except Exception as e:
        print("Nominatim indisponible:", repr(e))

    try:
        loc = _try_geocode_arc(test_query)
        if loc:
            print("Service gÃ©ocodage: ArcGIS")
            return "arc"
    except Exception as e:
        print("ArcGIS indisponible:", repr(e))

    return None


def obtenir_gps(adresse, service: str):
    if pd.isna(adresse):
        return None, None

    adresse = str(adresse).strip()
    if not adresse:
        return None, None

    key = adresse.lower()
    if key in _cache_gps:
        return _cache_gps[key]

    query = f"{adresse}, Tunis, Tunisia"

    # Une seule requÃªte par adresse, avec gestion des erreurs
    try:
        if service == "osm":
            location = _try_geocode_osm(query)
        elif service == "arc":
            location = _try_geocode_arc(query)
        else:
            location = None

        if location:
            res = (location.latitude, location.longitude)
            _cache_gps[key] = res
            return res
    except GeocoderRateLimited as e:
        # Si Ã§a rate-limit, on attend (si possible) et on abandonne pour cette adresse
        wait_s = (getattr(e, "retry_after", None) or 60) + 1
        print(f"Rate limited (429). Pause {wait_s}s...")
        time.sleep(wait_s)
    except (GeocoderTimedOut, GeocoderUnavailable):
        pass
    except GeocoderServiceError:
        pass

    _cache_gps[key] = (None, None)
    return None, None


# --- GÃ©ocodage optimisÃ©: uniquement les adresses uniques ---
adresses = df["Adresse destinataire"].fillna("").astype(str).str.strip()
adresses_non_vides = adresses[adresses != ""]
uniques = pd.unique(adresses_non_vides)

print("Nb adresses non vides:", len(adresses_non_vides))
print("Nb adresses uniques:", len(uniques))

service = choisir_service()

if service is None:
    print("Aucun service de gÃ©ocodage disponible. df_map restera vide.")
else:
    print("GÃ©ocodage en cours...")
    for i, adr in enumerate(uniques, start=1):
        obtenir_gps(adr, service)
        if i % 10 == 0:
            print(f"Progress: {i}/{len(uniques)} adresses gÃ©ocodÃ©es")


def _lat_lon_from_cache(adr):
    adr = str(adr).strip()
    if not adr:
        return None, None
    return _cache_gps.get(adr.lower(), (None, None))


df[["lat", "lon"]] = adresses.apply(lambda x: pd.Series(_lat_lon_from_cache(x)))

df_map = df.dropna(subset=["lat", "lon"]).copy()
print(f"SuccÃ¨s : {len(df_map)} adresses localisÃ©es sur {len(df)}.")

if len(df_map) == 0:
    exemples = uniques[:10].tolist() if len(uniques) else []
    print("Exemples d'adresses (10 premiÃ¨res uniques):", exemples)
    print("Si tout est Ã  0, vÃ©rifie: connexion Internet / VPN / firewall.")

Colonnes: ['Reference', 'Expéditeur', 'Adresse destinataire', 'Action', 'Contenu', 'Poids', 'Date de création', 'Statut', 'Gouvernorat', 'Prix Total']
Nb lignes: 77
Nb adresses non vides: 77
Nb adresses uniques: 40
Service gÃ©ocodage: Nominatim (OSM)
GÃ©ocodage en cours...
Progress: 10/40 adresses gÃ©ocodÃ©es
Progress: 20/40 adresses gÃ©ocodÃ©es
Progress: 30/40 adresses gÃ©ocodÃ©es
Progress: 40/40 adresses gÃ©ocodÃ©es
SuccÃ¨s : 32 adresses localisÃ©es sur 77.


In [22]:
df

,Reference,Expéditeur,Adresse destinataire,Action,Contenu,Poids,Date de création,Statut,Gouvernorat,Prix Total,lat,lon
0,171025500386,Boutique,Jeanne d'arc,NaN,Vêtements,0.8,2025-10-17T19:05:02,Livrée,Grand Tunis,241.0,36.820021,10.180950
1,10925307816,Boutique,ezzahra,NaN,Brume,0.3,2025-09-01T11:12:39,Livrée,Grand Tunis,100.0,36.869241,10.348213
2,10925789121,Boutique,Bellini café,NaN,Vêtements,0.8,2025-09-01T08:50:54,Livrée,Grand Tunis,0.0,NaN,NaN
3,10925486647,Boutique,Ennaser 2,NaN,Vêtements,1.0,2025-09-01T07:16:51,Livrée,Grand Tunis,0.0,NaN,NaN
4,290825175632,Boutique,Kram,NaN,Chaussures,1.5,2025-08-29T10:35:53,Livrée,Grand Tunis,275.0,36.847999,10.294355
...,...,...,...,...,...,...,...,...,...,...,...,...
72,250525270567,Boutique1,technopole el ghazela,NaN,vetements,0.9,2025-05-25T13:36:22,Livrée,Grand Tunis,0.0,NaN,NaN
73,210425541225,Boutique1,Sidi bousaid,NaN,Vêtements,1.0,2025-04-21T09:05:39,Livrée,Grand Tunis,10.0,36.768679,10.113587
74,130425400219,Boutique1,Ezzahra nakhil,NaN,vetements,1.4,2025-04-13T09:05:01,Livrée,Grand Tunis,400.0,NaN,NaN
75,60425552333,Boutique1,carrefour el marsa,NaN,casque blanc,0.5,2025-04-06T10:42:45,Livrée,Grand Tunis,33.0,36.866826,10.297551


In [23]:
import pandas as pd

def estimer_poids(contenu):
    """
    Estime le poids d'un colis en kg en fonction des mots-clÃ©s
    trouvÃ©s dans la description du contenu.
    """
    # Si le contenu est vide
    if pd.isna(contenu) or contenu == "":
        return 0.5 # Poids moyen par dÃ©faut


    texte = str(contenu).lower()

    # Dictionnaire de correspondance Mot-clÃ© -> Poids (en kg)

    categories = {
        'sac': 1.5,
        'talon': 1.2,
        'chaussures': 1.3,
        'coffret': 2.0,
        'burkini': 0.7,
        'vÃªtements': 0.8,
        'robe': 0.6,
        'accessoires': 0.3,
        'brume': 0.4,
        'makeup': 0.2,
        'maquillage': 0.2,
        'skin care': 0.5,
        'produit': 0.4,
        'casque': 0.5,
        'lunettes': 0.2
    }

    # On parcourt le dictionnaire pour voir si un mot-clÃ© est prÃ©sent
    for mot, poids in categories.items():
        if mot in texte:
            return poids

    # Si aucun mot n'est reconnu, on retourne un poids par dÃ©faut
    return 0.5

# --- EXEMPLE D'APPLICATION SUR TON DATAFRAME ---
# dfc_propre['Poids'] = dfc_propre['Contenu'].apply(estimer_poids)

In [24]:
import numpy as np
import random
from datetime import datetime, timedelta
import pandas as pd

def is_terrestre_tunis(lat, lon):
    """VÃ©rifie si les coordonnÃ©es sont sur la terre ferme dans le Grand Tunis."""
    # BoÃ®te englobante approximative du Grand Tunis
    if not (36.50 <= lat <= 37.00 and 9.80 <= lon <= 10.40):
        return False
    
    # Filtrer la MÃ©diterranÃ©e (La Marsa, Carthage, Gammarth, etc.)
    # La cÃ´te est en diagonale approximative
    if lat > 36.85 and lon > 10.34:
        return False
    if lat > 36.88 and lon > 10.33:
        return False
    if lat > 36.90 and lon > 10.31:
        return False
    if lat > 36.92 and lon > 10.28:
        return False
        
    # Filtrer le Lac de Tunis
    if (36.80 <= lat <= 36.84) and (10.22 <= lon <= 10.29):
        return False
        
    # Filtrer le Golfe de Tunis au sud de RadÃ¨s
    if lat < 36.75 and lon > 10.30:
        return False

    return True

def generer_date_aleatoire(date_base="2025-11-01"):
    """GÃ©nÃ¨re une heure alÃ©atoire entre 08:00 et 18:00"""
    heure = random.randint(8, 17)
    minute = random.randint(0, 59)
    seconde = random.randint(0, 59)

    date_obj = datetime.strptime(date_base, "%Y-%m-%d") + timedelta(
        hours=heure, minutes=minute, seconds=seconde
    )
    return date_obj.strftime("%Y-%m-%dT%H:%M:%S+01:00")

def generer_donnees_supp(df_reel, n_points=3):
    """GÃ©nÃ¨re des points supplÃ©mentaires Ã  partir de donnÃ©es rÃ©elles."""
    if df_reel is None or df_reel.empty:
        return pd.DataFrame()

    nouveaux_colis = []
    
    for _ in range(n_points):
        base = df_reel.sample(1).iloc[0]

        # Variation GPS (environ 1.5 km de variation max autour du point)
        # On essaie plusieurs fois jusqu'Ã  trouver un point valide terrestre
        for _ in range(15):
            lat_v = base["lat"] + np.random.uniform(-0.015, 0.015)
            lon_v = base["lon"] + np.random.uniform(-0.015, 0.015)
            if is_terrestre_tunis(lat_v, lon_v):
                break

        # Simulation Contenu et Poids
        contenu = random.choice(
            ["VÃªtements", "Accessoires", "Chaussures", "Coffret", "Skin Care", "Ã‰lectronique", "Livres"]
        )
        poids = estimer_poids(contenu)

        nouveaux_colis.append(
            {
                "Reference": f"{random.randint(100000000000, 999999999999)}",
                "ExpÃ©diteur": "Boutique",
                "Adresse destinataire": f"Zone {base['Adresse destinataire']}",
                "Contenu": contenu,
                "Poids": poids,
                "Date de crÃ©ation": generer_date_aleatoire(),
                "lat": lat_v,
                "lon": lon_v,
                "Gouvernorat": "Grand Tunis",
                "Statut": random.choice(["En attente", "En cours", "PrÃªt pour livraison"]),
            }
        )

    return pd.DataFrame(nouveaux_colis)

def enrichir_colonnes(df):
    """Ajoute de nouvelles colonnes basÃ©es sur les donnÃ©es existantes."""
    df = df.copy()
    
    # 1. Ajout des Frais de Livraison (7 TND de base + 1.5 TND par Kg)
    poids_num = pd.to_numeric(df["Poids"], errors='coerce').fillna(0.5)
    df["Frais_Livraison_TND"] = (7.0 + (poids_num * 1.5)).round(2)
    
    # 2. Ajout de la PrioritÃ© de la commande (Express pour les petits colis, etc.)
    conditions = [
        poids_num <= 0.5,
        poids_num >= 2.0
    ]
    choices = ["Express", "Basique"]
    df["Priorite"] = np.select(conditions, choices, default="Standard")
    
    # 3. Ajout du type de Paiement
    df["Mode_Paiement"] = np.random.choice(
        ["Paiement Ã  la livraison", "Carte Bancaire", "Virement"], 
        size=len(df), 
        p=[0.7, 0.2, 0.1]
    )
    
    # 4. Ajout du Type de Client
    df["Type_Client"] = np.random.choice(
        ["B2C (Particulier)", "B2B (Professionnel)"], 
        size=len(df), 
        p=[0.85, 0.15]
    )
    
    return df

# GÃ©nÃ©rer un petit nombre de donnÃ©es (pour arriver Ã  ~20 au total)
df1 = generer_donnees_supp(df_map, 8)

# Fusionner les donnÃ©es rÃ©elles et les donnÃ©es gÃ©nÃ©rÃ©es
dfc = pd.concat([df_map, df1], ignore_index=True)

# Nettoyer Ã©galement dfc de tout point potentiellement dans l'eau
masque_terrestre = dfc.apply(lambda row: is_terrestre_tunis(row["lat"], row["lon"]), axis=1)
dfc = dfc[masque_terrestre].copy()

# Enrichir avec les nouvelles colonnes
dfc = enrichir_colonnes(dfc)

# Trier par date pour la simulation chronologique
if "Date de crÃ©ation" in dfc.columns:
    dfc["Date de crÃ©ation"] = dfc["Date de crÃ©ation"].fillna(generer_date_aleatoire())
    dfc = dfc.sort_values(by="Date de crÃ©ation")

print(f"Total des commandes ajustÃ© Ã  : {len(dfc)} lignes.")
print(f"Nouvelles colonnes crÃ©Ã©es : {list(dfc.columns)}")

Total des commandes ajustÃ© Ã  : 36 lignes.
Nouvelles colonnes crÃ©Ã©es : ['Reference', 'Expéditeur', 'Adresse destinataire', 'Action', 'Contenu', 'Poids', 'Date de création', 'Statut', 'Gouvernorat', 'Prix Total', 'lat', 'lon', 'ExpÃ©diteur', 'Date de crÃ©ation', 'Frais_Livraison_TND', 'Priorite', 'Mode_Paiement', 'Type_Client']


In [25]:
dfc

,Reference,Expéditeur,Adresse destinataire,Action,Contenu,Poids,Date de création,Statut,Gouvernorat,Prix Total,lat,lon,ExpÃ©diteur,Date de crÃ©ation,Frais_Livraison_TND,Priorite,Mode_Paiement,Type_Client
36,195930661237,NaN,Zone lycée abou kacem chebbi,NaN,Chaussures,1.3,NaN,PrÃªt pour livraison,Grand Tunis,NaN,36.803261,10.160965,Boutique,2025-11-01T08:30:30+01:00,8.95,Standard,Paiement Ã la livraison,B2C (Particulier)
38,758968705664,NaN,Zone carrefour el marsa,NaN,Coffret,2.0,NaN,En attente,Grand Tunis,NaN,36.862658,10.292583,Boutique,2025-11-01T11:54:02+01:00,10.00,Basique,Paiement Ã la livraison,B2C (Particulier)
34,434597812226,NaN,Zone tunis lafayette,NaN,Coffret,2.0,NaN,PrÃªt pour livraison,Grand Tunis,NaN,36.828668,10.193049,Boutique,2025-11-01T14:16:45+01:00,10.00,Basique,Carte Bancaire,B2C (Particulier)
37,218712428537,NaN,Zone tunis centre ville,NaN,Ã‰lectronique,0.5,NaN,PrÃªt pour livraison,Grand Tunis,NaN,36.806322,10.183807,Boutique,2025-11-01T14:35:29+01:00,7.75,Express,Paiement Ã la livraison,B2C (Particulier)
33,974434736764,NaN,Zone zahrouni,NaN,VÃªtements,0.5,NaN,En cours,Grand Tunis,NaN,36.778066,10.118835,Boutique,2025-11-01T15:29:53+01:00,7.75,Express,Paiement Ã la livraison,B2B (Professionnel)
39,642621890421,NaN,Zone Kram,NaN,Coffret,2.0,NaN,En attente,Grand Tunis,NaN,36.849047,10.284133,Boutique,2025-11-01T15:33:05+01:00,10.00,Basique,Paiement Ã la livraison,B2C (Particulier)
32,325601650785,NaN,Zone manouba manouba,NaN,Skin Care,0.5,NaN,En cours,Grand Tunis,NaN,36.765862,10.015760,Boutique,2025-11-01T16:47:56+01:00,7.75,Express,Paiement Ã la livraison,B2C (Particulier)
35,805521115610,NaN,Zone Sidi bousaid,NaN,Chaussures,1.3,NaN,PrÃªt pour livraison,Grand Tunis,NaN,36.781280,10.124399,Boutique,2025-11-01T17:27:05+01:00,8.95,Standard,Paiement Ã la livraison,B2B (Professionnel)
31,60425504211,Boutique1,jardins d'el menzah 1,NaN,swim set blanc,0.4,2025-04-06T11:34:12,Livrée,Grand Tunis,33.0,36.835388,10.166722,NaN,2025-11-01T17:40:38+01:00,7.60,Express,Paiement Ã la livraison,B2C (Particulier)
20,171025500386,Boutique,Jeanne d'arc,NaN,Vêtements,0.8,2025-10-17T19:05:02,Livrée,Grand Tunis,241.0,36.820021,10.180950,NaN,2025-11-01T17:40:38+01:00,8.20,Standard,Paiement Ã la livraison,B2C (Particulier)


In [31]:
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="cleaner_tunis")



lon_max = 10.32
lat_max = 36.92
lat_min = 36.70

# Filtrage mathÃ©matique simple
dfc_propre_filtre = dfc[
    (dfc['lon'] <= lon_max) &
    (dfc['lat'] <= lat_max) &
    (dfc['lat'] >= lat_min)
].copy()

# On s'assure d'avoir au moins 20 points, mÃªme si certains sont hors cadre
if len(dfc_propre_filtre) < 20 and len(dfc) >= 20:
    print("Le filtrage a trop rÃ©duit le nombre de points. On ajuste pour en garder 20.")
    dfc_propre = dfc.head(20).copy()
else:
    dfc_propre = dfc_propre_filtre

print(f"Ancien nombre de points : {len(dfc)}")
print(f"Nouveau nombre de points (aprÃ¨s ajustement) : {len(dfc_propre)}")

Ancien nombre de points : 36
Nouveau nombre de points (aprÃ¨s ajustement) : 36


In [27]:
dfc_propre

,Reference,Expéditeur,Adresse destinataire,Action,Contenu,Poids,Date de création,Statut,Gouvernorat,Prix Total,lat,lon,ExpÃ©diteur,Date de crÃ©ation,Frais_Livraison_TND,Priorite,Mode_Paiement,Type_Client
36,195930661237,NaN,Zone lycée abou kacem chebbi,NaN,Chaussures,1.3,NaN,PrÃªt pour livraison,Grand Tunis,NaN,36.803261,10.160965,Boutique,2025-11-01T08:30:30+01:00,8.95,Standard,Paiement Ã la livraison,B2C (Particulier)
38,758968705664,NaN,Zone carrefour el marsa,NaN,Coffret,2.0,NaN,En attente,Grand Tunis,NaN,36.862658,10.292583,Boutique,2025-11-01T11:54:02+01:00,10.00,Basique,Paiement Ã la livraison,B2C (Particulier)
34,434597812226,NaN,Zone tunis lafayette,NaN,Coffret,2.0,NaN,PrÃªt pour livraison,Grand Tunis,NaN,36.828668,10.193049,Boutique,2025-11-01T14:16:45+01:00,10.00,Basique,Carte Bancaire,B2C (Particulier)
37,218712428537,NaN,Zone tunis centre ville,NaN,Ã‰lectronique,0.5,NaN,PrÃªt pour livraison,Grand Tunis,NaN,36.806322,10.183807,Boutique,2025-11-01T14:35:29+01:00,7.75,Express,Paiement Ã la livraison,B2C (Particulier)
33,974434736764,NaN,Zone zahrouni,NaN,VÃªtements,0.5,NaN,En cours,Grand Tunis,NaN,36.778066,10.118835,Boutique,2025-11-01T15:29:53+01:00,7.75,Express,Paiement Ã la livraison,B2B (Professionnel)
39,642621890421,NaN,Zone Kram,NaN,Coffret,2.0,NaN,En attente,Grand Tunis,NaN,36.849047,10.284133,Boutique,2025-11-01T15:33:05+01:00,10.00,Basique,Paiement Ã la livraison,B2C (Particulier)
32,325601650785,NaN,Zone manouba manouba,NaN,Skin Care,0.5,NaN,En cours,Grand Tunis,NaN,36.765862,10.015760,Boutique,2025-11-01T16:47:56+01:00,7.75,Express,Paiement Ã la livraison,B2C (Particulier)
35,805521115610,NaN,Zone Sidi bousaid,NaN,Chaussures,1.3,NaN,PrÃªt pour livraison,Grand Tunis,NaN,36.781280,10.124399,Boutique,2025-11-01T17:27:05+01:00,8.95,Standard,Paiement Ã la livraison,B2B (Professionnel)
31,60425504211,Boutique1,jardins d'el menzah 1,NaN,swim set blanc,0.4,2025-04-06T11:34:12,Livrée,Grand Tunis,33.0,36.835388,10.166722,NaN,2025-11-01T17:40:38+01:00,7.60,Express,Paiement Ã la livraison,B2C (Particulier)
20,171025500386,Boutique,Jeanne d'arc,NaN,Vêtements,0.8,2025-10-17T19:05:02,Livrée,Grand Tunis,241.0,36.820021,10.180950,NaN,2025-11-01T17:40:38+01:00,8.20,Standard,Paiement Ã la livraison,B2C (Particulier)


In [32]:
import json
import html
from pathlib import Path

# Optionnel: si tu as une vraie clÃ© Google Maps JavaScript API, mets-la ici.
# Sinon, le notebook gÃ©nÃ¨re une solution Google Maps SANS clÃ© (KML + liens + page HTML).
GOOGLE_MAPS_API_KEY = ""  # ex: "AIza...." (laisser vide si tu n'as pas de clÃ©)

# On utilise dfc_propre (la grosse base augmentÃ©e) au lieu de df_map (seulement les originaux)
if "dfc_propre" not in globals() or dfc_propre is None or dfc_propre.empty:
    raise ValueError("dfc_propre is empty. Veuillez gÃ©nÃ©rer les donnÃ©es supplÃ©mentaires en premier.")

points = dfc_propre.dropna(subset=["lat", "lon"]).copy()

markers = []
for _, row in points.iterrows():
    ref = "" if pd.isna(row.get("Reference")) else str(row.get("Reference"))
    adr = "" if pd.isna(row.get("Adresse destinataire")) else str(row.get("Adresse destinataire"))
    statut = "" if pd.isna(row.get("Statut")) else str(row.get("Statut"))
    
    # On peut aussi ajouter les nouvelles colonnes dans l'affichage !
    frais = f"{row.get('Frais_Livraison_TND', 'N/A')} TND"
    priorite = "" if pd.isna(row.get("Priorite")) else str(row.get("Priorite"))

    title = f"{ref} - {adr}".strip(" -")
    info_html = (
        f"<div>"
        f"<b>RÃ©fÃ©rence:</b> {html.escape(ref)}<br/>"
        f"<b>Adresse:</b> {html.escape(adr)}<br/>"
        f"<b>Frais:</b> {html.escape(frais)}<br/>"
        f"<b>PrioritÃ©:</b> {html.escape(priorite)}<br/>"
        f"<b>Statut:</b> {html.escape(statut)}"
        f"</div>"
    )

    markers.append(
        {
            "lat": float(row["lat"]),
            "lng": float(row["lon"]),
            "title": title,
            "infoHtml": info_html,
        }
    )

# --- Ajout du point de l'entrepÃ´t ---
markers.append(
    {
        "lat": 36.82857,
        "lng": 10.20616,
        "title": "EntrepÃ´t Principal",
        "infoHtml": (
            "<div>"
            "<b>EntrepÃ´t Principal</b><br/>"
            "Point de dÃ©part des livraisons"
            "</div>"
        ),
    }
)

# ----------------------------
# 1) Export KML (Google My Maps)
# ----------------------------
# Google My Maps accepte l'import KML (sans API key).

kml_path = Path("livraisons_mymaps.kml")

placemarks = []
for m in markers:
    name = html.escape(m["title"] or "Destination")
    desc = m["infoHtml"]
    lon = m["lng"]
    lat = m["lat"]
    placemarks.append(
        "\n".join(
            [
                "<Placemark>",
                f"  <name>{name}</name>",
                f"  <description><![CDATA[{desc}]]></description>",
                "  <Point>",
                f"    <coordinates>{lon},{lat},0</coordinates>",
                "  </Point>",
                "</Placemark>",
            ]
        )
    )

kml_doc = "\n".join(
    [
        "<?xml version=\"1.0\" encoding=\"UTF-8\"?>",
        "<kml xmlns=\"http://www.opengis.net/kml/2.2\">",
        "<Document>",
        "  <name>Livraisons</name>",
        *placemarks,
        "</Document>",
        "</kml>",
        "",
    ]
)

kml_path.write_text(kml_doc, encoding="utf-8")

# ------------------------------------
# 2) HTML avec liens Google Maps (sans clÃ©)
# ------------------------------------
# Chaque lien ouvre un point directement dans Google Maps.

links_path = Path("google_maps_links.html")

rows = []
for m in markers:
    q = f"{m['lat']},{m['lng']}"
    url = f"https://www.google.com/maps/search/?api=1&query={q}"
    label = m["title"] or q
    rows.append(
        f"<li><a href=\"{url}\" target=\"_blank\" rel=\"noopener\">{html.escape(label)}</a></li>"
    )

links_html = "\n".join(
    [
        "<!doctype html>",
        "<html>",
        "<head><meta charset=\"utf-8\"><title>Livraisons - Liens Google Maps</title></head>",
        "<body>",
        "<h2>Destinations (ouvrir dans Google Maps)</h2>",
        "<b>Total points:</b> " + str(len(markers)),
        "<ol>",
        *rows,
        "</ol>",
        "</body>",
        "</html>",
        "",
    ]
)

links_path.write_text(links_html, encoding="utf-8")

# -------------------------------------------------
# 3) Page principale: avec clÃ© -> carte interactive, sans clÃ© -> page d'instructions
# -------------------------------------------------

main_path = Path("google_maps_livraisons.html")

if GOOGLE_MAPS_API_KEY and "PASTE_YOUR_API_KEY_HERE" not in GOOGLE_MAPS_API_KEY:
    center_lat = float(points["lat"].mean())
    center_lng = float(points["lon"].mean())

    html_doc = f"""<!doctype html>
<html>
  <head>
    <meta charset=\"utf-8\" />
    <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />
    <title>Livraisons - Google Maps</title>
    <style>
      html, body, #map {{ height: 100%; margin: 0; padding: 0; }}
    </style>
  </head>
  <body>
    <div id=\"map\"></div>

    <script>
      const markers = {json.dumps(markers, ensure_ascii=False)};

      function initMap() {{
        const center = {{ lat: {center_lat}, lng: {center_lng} }};
        const map = new google.maps.Map(document.getElementById('map'), {{
          zoom: 11,
          center
        }});

        const bounds = new google.maps.LatLngBounds();
        const infoWindow = new google.maps.InfoWindow();

        for (const m of markers) {{
          const pos = new google.maps.LatLng(m.lat, m.lng);
          bounds.extend(pos);

          const marker = new google.maps.Marker({{
            position: pos,
            map,
            title: m.title
          }});

          marker.addListener('click', () => {{
            infoWindow.setContent(m.infoHtml);
            infoWindow.open(map, marker);
          }});
        }}

        if (markers.length > 1) {{
          map.fitBounds(bounds);
        }}
      }}
    </script>

    <script async defer
      src=\"https://maps.googleapis.com/maps/api/js?key={GOOGLE_MAPS_API_KEY}&callback=initMap\">
    </script>
  </body>
</html>
"""

    main_path.write_text(html_doc, encoding="utf-8")
    print(f"Saved (JS API map): {main_path.resolve()}")
else:
    # IMPORTANT: sans clÃ©, on ne charge PAS l'API Google Maps JS (sinon InvalidKeyMapError)
    instructions = "\n".join(
        [
            "<!doctype html>",
            "<html>",
            "<head><meta charset=\"utf-8\"><title>Livraisons - Google Maps</title></head>",
            "<body>",
            "<h2>Livraisons (Google Maps) â€” sans clÃ© API</h2>",
            "<p>La carte Google Maps interactive nÃ©cessite une clÃ© API valide. Pour afficher tes " + str(len(markers)) + " points sur Google Maps <b>sans clÃ©</b>:</p>",
            "<ol>",
            "  <li>Va sur <a href=\"https://www.google.com/mymaps\" target=\"_blank\" rel=\"noopener\">Google My Maps</a></li>",
            "  <li>CrÃ©e une nouvelle carte</li>",
            "  <li>Clique sur <b>Importer</b> et choisis le fichier <b>livraisons_mymaps.kml</b></li>",
            "</ol>",
            "<p>Ou ouvre directement chaque point via la liste de liens:</p>",
            "<p><a href=\"google_maps_links.html\">Ouvrir google_maps_links.html</a></p>",
            "</body>",
            "</html>",
            "",
        ]
    )
    main_path.write_text(instructions, encoding="utf-8")
    print(f"Saved (no-key instructions page): {main_path.resolve()}")

print(f"Saved (My Maps import with {len(markers)} points) : {kml_path.resolve()}")
print(f"Saved (Links): {links_path.resolve()}")

Saved (no-key instructions page): C:\Users\ghayd\Documents\p2m\V-2-V--logistics-main\central agent\google_maps_livraisons.html
Saved (My Maps import with 37 points) : C:\Users\ghayd\Documents\p2m\V-2-V--logistics-main\central agent\livraisons_mymaps.kml
Saved (Links): C:\Users\ghayd\Documents\p2m\V-2-V--logistics-main\central agent\google_maps_links.html


In [29]:
%pip install ortools

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


optimisation statique  

In [33]:
import pandas as pd
import numpy as np
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
from geopy.distance import geodesic

# =================================================================
# 1. PARAMÈTRES LOGISTIQUES ET CONFIGURATION
# =================================================================
VITESSE_MOYENNE_KMH = 30 
TEMPS_STOP_LIVRAISON_SEC = 10 * 60  # 10 minutes par client
TEMPS_TRAVAIL_TOTAL = 9 * 3600     # 9 heures max de contrat
MARGE_RESERVE = 3 * 3600           # 3 heures de réserve vide pour nouveaux colis
TEMPS_PLANIFIABLE_MAX = TEMPS_TRAVAIL_TOTAL - MARGE_RESERVE # Limite à 6h de route/stop

CAP_CAMION_KG = 50                 # Capacité max du véhicule
ENTREPOT_COORD = (36.82857, 10.20616)

# =================================================================
# 2. PRÉPARATION DES MATRICES (DISTANCE & TEMPS)
# =================================================================
def calculer_matrices_logistiques(df):
    points = [ENTREPOT_COORD] + list(zip(df['lat'], df['lon']))
    n = len(points)
    dist_matrix = np.zeros((n, n), dtype=int)
    time_matrix = np.zeros((n, n), dtype=int)
    
    for i in range(n):
        for j in range(n):
            if i == j: continue
            # Calcul distance réelle (mètres)
            d_metres = geodesic(points[i], points[j]).meters
            dist_matrix[i][j] = int(d_metres)
            
            # Temps = Trajet + 10min de stop si destination = client
            temps_trajet = int((d_metres / 1000) / VITESSE_MOYENNE_KMH * 3600)
            temps_stop = TEMPS_STOP_LIVRAISON_SEC if j > 0 else 0
            time_matrix[i][j] = temps_trajet + temps_stop
            
    return dist_matrix, time_matrix

# =================================================================
# 3. MOTEUR D'OPTIMISATION (VRP)
# =================================================================
def resoudre_tournees(n_chauffeurs, dist_m, time_m, poids_colis, duree_vie_colis):
    manager = pywrapcp.RoutingIndexManager(len(dist_m), n_chauffeurs, 0)
    routing = pywrapcp.RoutingModel(manager)

    # RENTABILITÉ : Minimiser la distance (Carburant)
    def distance_callback(from_idx, to_idx):
        return int(dist_m[manager.IndexToNode(from_idx)][manager.IndexToNode(to_idx)])
    transit_idx = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_idx)

    # CONTRAINTE TEMPS : Max 6h (Trajet + Stops)
    def time_callback(from_idx, to_idx):
        return int(time_m[manager.IndexToNode(from_idx)][manager.IndexToNode(to_idx)])
    time_idx = routing.RegisterTransitCallback(time_callback)
    routing.AddDimension(time_idx, 3600, TEMPS_PLANIFIABLE_MAX, True, 'Time')
    time_dimension = routing.GetDimensionOrDie('Time')

    # DURÉE DE VIE : Respecter les deadlines de chaque colis
    for node_idx, deadline in enumerate(duree_vie_colis):
        if node_idx == 0: continue
        index = manager.NodeToIndex(node_idx)
        time_dimension.CumulVar(index).SetRange(0, int(deadline))

    # CAPACITÉ : Max 50kg
    def demand_callback(from_idx):
        return int(poids_colis[manager.IndexToNode(from_idx)])
    demand_idx = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(demand_idx, 0, [CAP_CAMION_KG]*n_chauffeurs, True, 'Capacity')

    # Paramètres IA
    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_params.time_limit.seconds = 5
    
    return manager, routing, routing.SolveWithParameters(search_params)

# =================================================================
# 4. EXÉCUTION : RECHERCHE DE LA MEILLEURE FLOTTE
# =================================================================

# Nettoyage et préparation
poids_data = [0] + list(pd.to_numeric(dfc_propre['Poids'], errors='coerce').fillna(1).astype(int))
# Par défaut, durée de vie de 6h pour tous (modifiable selon urgence)
deadlines_data = [TEMPS_PLANIFIABLE_MAX] * (len(dfc_propre) + 1)

dist_m, time_m = calculer_matrices_logistiques(dfc_propre)

flotte_optimale = None
print("🤖 Analyse de rentabilité en cours...")

# Test de 1 à 15 chauffeurs pour justifier le nombre minimal
for n in range(1, 16):
    man, rout, sol = resoudre_tournees(n, dist_m, time_m, poids_data, deadlines_data)
    if sol:
        flotte_optimale = (man, rout, sol, n)
        print(f"✅ SOLUTION OPTIMALE : {n} chauffeurs suffisent pour respecter les 6h + 3h de marge.")
        break


# =================================================================
# 5. AFFICHAGE ÉPURÉ : UNIQUEMENT LES LIENS GPS PAR CHAUFFEUR
# =================================================================

if flotte_optimale:
    man, rout, sol, n_final = flotte_optimale
    
    print(f"\n🚀 {n_final} CHAUFFEURS MOBILISÉS - RÉSUMÉ DES ITINÉRAIRES")
    print("="*80)

    for v_id in range(n_final):
        index = rout.Start(v_id)
        
        # On vérifie si le chauffeur a des colis (on ignore les camions vides)
        if rout.IsEnd(sol.Value(rout.NextVar(index))):
            continue

        # Liste des coordonnées : Départ Entrepôt
        gps_coords = [f"{ENTREPOT_COORD[0]},{ENTREPOT_COORD[1]}"]
        nb_colis = 0
        dist_v = 0
        
        # On parcourt le trajet pour extraire les coordonnées et la distance
        while not rout.IsEnd(index):
            node_idx = man.IndexToNode(index)
            if node_idx != 0:
                nb_colis += 1
                client = dfc_propre.iloc[node_idx-1]
                gps_coords.append(f"{client['lat']},{client['lon']}")
            
            prev_idx = index
            index = sol.Value(rout.NextVar(index))
            dist_v += rout.GetArcCostForVehicle(prev_idx, index, v_id)
        
        # Retour à l'entrepôt
        gps_coords.append(f"{ENTREPOT_COORD[0]},{ENTREPOT_COORD[1]}")
        
        # Calcul du temps total (incluant les stops de 10 min)
        t_total_h = sol.Value(rout.GetDimensionOrDie('Time').CumulVar(rout.End(v_id))) / 3600
        marge_h = 9.0 - t_total_h

        # --- AFFICHAGE ULTRA-CONCIS ---
        print(f"\n🚛 CHAUFFEUR {v_id + 1} ({nb_colis} colis | {dist_v/1000:.1f} km)")
        print(f"⏱️  Travail : {t_total_h:.2f}h | Marge de sécurité : {marge_h:.2f}h")
        
        # Lien Google Maps multi-points
        url_gps = "https://www.google.com/maps/dir/" + "/".join(gps_coords)
        print(f"📍 GPS : {url_gps}")
        print("-" * 80)

  
    


🤖 Analyse de rentabilité en cours...
✅ SOLUTION OPTIMALE : 2 chauffeurs suffisent pour respecter les 6h + 3h de marge.

🚀 2 CHAUFFEURS MOBILISÉS - RÉSUMÉ DES ITINÉRAIRES

🚛 CHAUFFEUR 1 (10 colis | 22.0 km)
⏱️  Travail : 2.40h | Marge de sécurité : 6.60h
📍 GPS : https://www.google.com/maps/dir/36.82857,10.20616/36.8620857,10.2108277/36.8620857,10.2108277/36.8620857,10.2108277/36.8620857,10.2108277/36.8668257,10.2975514/36.8668257,10.2975514/36.86265823979696,10.292583124450449/36.8479991,10.2943555/36.8479991,10.2943555/36.849047016913914,10.28413265181166/36.82857,10.20616
--------------------------------------------------------------------------------

🚛 CHAUFFEUR 2 (26 colis | 44.2 km)
⏱️  Travail : 5.81h | Marge de sécurité : 3.19h
📍 GPS : https://www.google.com/maps/dir/36.82857,10.20616/36.8328463,10.1995966/36.8328463,10.1995966/36.8353876,10.1667221/36.8353876,10.1667221/36.8372507,10.1402003/36.8372507,10.1402003/36.7922783,10.1078508/36.7922783,10.1078508/36.76586229565319,10.

calcul de rentabilite par flotte ,

In [ ]:
# =============================================================================
# CELLULE NOTEBOOK — SCÉNARIO B COMPLET
# Branchée directement sur dfc_propre + solution OR-Tools existante
#
# Colonnes utilisées depuis dfc_propre :
#   Reference, Adresse destinataire, Poids, Date de création,
#   Frais_Livraison_TND, Priorite, Mode_Paiement, Type_Client, lat, lon
#
# Prérequis : avoir exécuté les cellules précédentes qui produisent
#   - dfc_propre      : DataFrame nettoyé
#   - flotte_optimale : tuple (manager, routing, solution, n_chauffeurs)
# =============================================================================
 
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from geopy.distance import geodesic
 
# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTES
# ─────────────────────────────────────────────────────────────────────────────
VITESSE_KMH         = 30
TEMPS_STOP_SEC      = 10 * 60
TEMPS_MAX_SEC       = 9  * 3600
CAP_CAMION_KG       = 50
ENTREPOT_COORD      = (36.82857, 10.20616)
CONSOMMATION_L_KM   = 0.12
PRIX_CARBURANT_TND  = 2.1
TARIF_LIVRAISON_TND = 8.0        # TND / colis — règle métier fallback
DUREE_VIE_H         = 24         # deadline = Date de création + 24h
HEURE_DEPART        = datetime.now().replace(hour=8, minute=0, second=0, microsecond=0)
 
 
# ─────────────────────────────────────────────────────────────────────────────
# UTILITAIRES
# ─────────────────────────────────────────────────────────────────────────────
 
def _dist_km(a, b):
    return geodesic(a, b).km
 
def _temps_sec(d_km, avec_stop=True):
    return (d_km / VITESSE_KMH) * 3600 + (TEMPS_STOP_SEC if avec_stop else 0)
 
def _deadline_sec(date_creation):
    """
    Convertit Date de création → deadline en secondes depuis HEURE_DEPART.
    Règle métier : deadline = Date de création + 24h.
    """
    if pd.isna(date_creation) if not isinstance(date_creation, datetime) else False:
        return TEMPS_MAX_SEC * 4
    try:
        if isinstance(date_creation, str):
            dt = pd.to_datetime(date_creation, utc=False).to_pydatetime()
            dt = dt.replace(tzinfo=None)
        elif isinstance(date_creation, datetime):
            dt = date_creation.replace(tzinfo=None)
        else:
            dt = pd.Timestamp(date_creation).to_pydatetime().replace(tzinfo=None)
        deadline_abs = dt + timedelta(hours=DUREE_VIE_H)
        delta = (deadline_abs - HEURE_DEPART).total_seconds()
        return max(delta, TEMPS_STOP_SEC)
    except Exception:
        return TEMPS_MAX_SEC * 4
 
 
# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 1 — ADAPTATEUR : OR-Tools solution → flotte Python native
#
# Lit dfc_propre + flotte_optimale et reconstruit pour chaque chauffeur :
#   sa tournée avec toutes les colonnes du dataset, son poids, sa distance
# ─────────────────────────────────────────────────────────────────────────────
 
def construire_flotte_depuis_ortools(dfc_propre, flotte_optimale):
    man, rout, sol, n_chauffeurs = flotte_optimale
    poids_series = pd.to_numeric(dfc_propre['Poids'], errors='coerce').fillna(1.0)
    flotte = []
 
    for v_id in range(n_chauffeurs):
        index = rout.Start(v_id)
        if rout.IsEnd(sol.Value(rout.NextVar(index))):
            continue  # chauffeur sans colis
 
        tournee   = []
        dist_km   = 0.0
        pos       = ENTREPOT_COORD
 
        while not rout.IsEnd(index):
            node       = man.IndexToNode(index)
            next_index = sol.Value(rout.NextVar(index))
 
            if node != 0:  # 0 = entrepôt
                row  = dfc_propre.iloc[node - 1]
                dest = (float(row['lat']), float(row['lon']))
                d    = _dist_km(pos, dest)
                dist_km += d
                pos      = dest
 
                tournee.append({
                    'id'           : str(row.get('Reference',              f'C{node}')),
                    'lat'          : float(row['lat']),
                    'lon'          : float(row['lon']),
                    'poids_kg'     : float(poids_series.iloc[node - 1]),
                    'deadline_sec' : _deadline_sec(row.get('Date de création')),
                    'date_creation': row.get('Date de création'),
                    'description'  : str(row.get('Adresse destinataire',   '')),
                    'priorite'     : str(row.get('Priorite',               'Standard')),
                    'mode_paiement': str(row.get('Mode_Paiement',          '')),
                    'type_client'  : str(row.get('Type_Client',            '')),
                    'contenu'      : str(row.get('Contenu',                '')),
                })
 
            index = next_index
 
        # retour entrepôt (distance uniquement, pas de stop)
        dist_km += _dist_km(pos, ENTREPOT_COORD)
 
        flotte.append({
            'id'               : v_id + 1,
            'nom'              : f'Chauffeur {v_id + 1}',
            'tournee'          : tournee,
            'poids_charge_kg'  : round(sum(c['poids_kg'] for c in tournee), 2),
            'distance_km'      : round(dist_km, 2),
            'temps_ecoule_sec' : 0.0,           # les camions viennent de partir
            'prochain_stop_idx': 0,              # aucune livraison encore effectuée
            'position_actuelle': ENTREPOT_COORD,
        })
 
    return flotte
 
 
# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 2 — RENTABILITÉ
# Utilise Frais_Livraison_TND réels de chaque colis (pas un tarif fixe)
# ─────────────────────────────────────────────────────────────────────────────
 
def rentabilite_chauffeur(ch):
    nb_colis       = len(ch['tournee'])
    revenus        = nb_colis * TARIF_LIVRAISON_TND
    cout_carburant = round(ch['distance_km'] * CONSOMMATION_L_KM * PRIX_CARBURANT_TND, 2)
    marge_brute    = round(revenus - cout_carburant, 2)
    marge_pct      = round(marge_brute / revenus * 100, 1) if revenus > 0 else 0.0
    return {
        'chauffeur'      : ch['nom'],
        'nb_colis'       : nb_colis,
        'revenus_tnd'    : round(revenus,        2),
        'distance_km'    : round(ch['distance_km'], 2),
        'cout_carburant' : cout_carburant,
        'marge_brute'    : marge_brute,
        'marge_pct'      : marge_pct,
    }
 
def rentabilite_flotte(flotte):
    bilans    = [rentabilite_chauffeur(ch) for ch in flotte]
    tot_colis = sum(b['nb_colis']       for b in bilans)
    tot_rev   = round(tot_colis * TARIF_LIVRAISON_TND, 2)
    tot_dist  = round(sum(b['distance_km']    for b in bilans), 2)
    tot_carb  = round(sum(b['cout_carburant'] for b in bilans), 2)
    tot_marge = round(tot_rev - tot_carb, 2)
    marge_pct = round(tot_marge / tot_rev * 100, 1) if tot_rev > 0 else 0.0
    return {
        'detail'          : bilans,
        'total_colis'     : tot_colis,
        'total_revenus'   : tot_rev,
        'total_distance'  : tot_dist,
        'total_carburant' : tot_carb,
        'total_marge'     : tot_marge,
        'marge_pct'       : marge_pct,
    }
 
def afficher_rentabilite(flotte):
    b         = rentabilite_flotte(flotte)
    SEP       = '─' * 82
    DBL       = '═' * 82
    icone_seuil = lambda p: '✅' if p >= 70 else ('⚠️ ' if p >= 50 else '🔴')
 
    print(f'\n{DBL}')
    print(f'  💰  BILAN DE RENTABILITÉ — {HEURE_DEPART.strftime("%d/%m/%Y")}')
    print(f'  Tarif     : {TARIF_LIVRAISON_TND} TND × nb colis livrés')
    print(f'  Carburant : {CONSOMMATION_L_KM*100:.0f} L/100km × {PRIX_CARBURANT_TND} TND/L')
    print(f'{DBL}')
    print(f"\n  {'Chauffeur':<18} {'Colis':>5}  {'Revenus':>11}  {'Distance':>10}  {'Carburant':>11}  {'Marge':>10}  {'%':>6}")
    print(f'  {SEP}')
 
    for r in b['detail']:
        print(
            f"  {r['chauffeur']:<18} {r['nb_colis']:>5}  "
            f"{r['revenus_tnd']:>9.2f} TND  "
            f"{r['distance_km']:>8.1f} km  "
            f"{r['cout_carburant']:>9.2f} TND  "
            f"{r['marge_brute']:>8.2f} TND  "
            f"{icone_seuil(r['marge_pct'])}{r['marge_pct']:>4.1f}%"
        )
 
    print(f'  {SEP}')
    print(
        f"  {'TOTAL FLOTTE':<18} {b['total_colis']:>5}  "
        f"{b['total_revenus']:>9.2f} TND  "
        f"{b['total_distance']:>8.1f} km  "
        f"{b['total_carburant']:>9.2f} TND  "
        f"{b['total_marge']:>8.2f} TND  "
        f"{icone_seuil(b['marge_pct'])}{b['marge_pct']:>4.1f}%"
    )
 
    barre = '█' * int(b['marge_pct'] / 100 * 42) + '░' * (42 - int(b['marge_pct'] / 100 * 42))
    print(f'\n  {DBL}')
    print(f"  📦 Colis livrés         : {b['total_colis']}  ×  {TARIF_LIVRAISON_TND} TND")
    print(f"  💵 Revenus totaux       : {b['total_revenus']:.2f} TND")
    print(f"  ⛽ Coût carburant total : {b['total_carburant']:.2f} TND  ({b['total_distance']:.1f} km)")
    print(f"  📈 Marge brute          : {b['total_marge']:.2f} TND")
    print(f"  {icone_seuil(b['marge_pct'])} Marge bénéficiaire   : {b['marge_pct']:.1f}%")
    print(f"  [{barre}]")
    print(f'  {DBL}\n')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ÉTAPE 3 — INSERTION DYNAMIQUE D'UN NOUVEAU COLIS
# Algorithme Cheapest Feasible Insertion (CFI)
# ─────────────────────────────────────────────────────────────────────────────

def _recalculer_planning(ch, tournee_candidate):
    """
    Simule le temps cumulé de la tournée candidate.
    Retourne (temps_total_sec, etas[]) où etas[i] = temps à l'arrivée au stop i.
    """
    t   = ch['temps_ecoule_sec']
    pos = ch['position_actuelle']
    etas = []
    for stop in tournee_candidate[ch['prochain_stop_idx']:]:
        d  = _dist_km(pos, (stop['lat'], stop['lon']))
        t += _temps_sec(d, avec_stop=True)
        etas.append(t)
        pos = (stop['lat'], stop['lon'])
    t += _dist_km(pos, ENTREPOT_COORD) / VITESSE_KMH * 3600
    return t, etas

def evaluer_insertion(ch, nouveau_colis, position):
    stops_restants = ch['tournee'][ch['prochain_stop_idx']:]

    # Contrainte 1 — poids
    poids_dispo = CAP_CAMION_KG - ch['poids_charge_kg']
    if nouveau_colis['poids_kg'] > poids_dispo:
        return {'ok': False, 'motif': f"Surpoids {nouveau_colis['poids_kg']}kg > {poids_dispo:.1f}kg dispo"}

    # Delta distance (carburant marginal)
    prev = ch['position_actuelle']                if position == 0                   else (stops_restants[position-1]['lat'], stops_restants[position-1]['lon'])
    nxt  = ENTREPOT_COORD                         if position == len(stops_restants) else (stops_restants[position]['lat'],   stops_restants[position]['lon'])
    new  = (nouveau_colis['lat'], nouveau_colis['lon'])

    delta_km   = _dist_km(prev, new) + _dist_km(new, nxt) - _dist_km(prev, nxt)
    delta_cout = round(delta_km * CONSOMMATION_L_KM * PRIX_CARBURANT_TND, 4)

    # Nouvelle séquence
    nouvelle_seq      = stops_restants[:position] + [nouveau_colis] + stops_restants[position:]
    tournee_candidate = ch['tournee'][:ch['prochain_stop_idx']] + nouvelle_seq
    temps_total, etas = _recalculer_planning(ch, tournee_candidate)
    eta_nouveau       = next(e for s, e in zip(nouvelle_seq, etas) if s['id'] == nouveau_colis['id'])

    # Contrainte 2 — deadline nouveau colis
    if eta_nouveau > nouveau_colis['deadline_sec']:
        return {'ok': False, 'motif': f"Deadline manquée : ETA {eta_nouveau/3600:.2f}h > {nouveau_colis['deadline_sec']/3600:.2f}h"}

    # Contrainte 3 — deadlines colis existants (l'insertion décale tout le monde)
    for stop, eta in zip(nouvelle_seq, etas):
        if stop['id'] != nouveau_colis['id'] and eta > stop['deadline_sec']:
            return {'ok': False, 'motif': f"Deadline de {stop['id']} violée : ETA {eta/3600:.2f}h"}

    # Contrainte 4 — temps de travail 9h
    if temps_total > TEMPS_MAX_SEC:
        return {'ok': False, 'motif': f"Dépasse 9h : +{(temps_total-TEMPS_MAX_SEC)/3600:.2f}h"}

    return {
        'ok'              : True,
        'delta_km'        : round(delta_km, 3),
        'delta_cout_tnd'  : delta_cout,
        'eta_nouveau_sec' : eta_nouveau,
        'temps_total_sec' : temps_total,
        'tournee_finale'  : tournee_candidate,
    }

def inserer_nouveau_colis(flotte, nouveau_colis,
                          poids_carburant=0.60,
                          poids_temps=0.25,
                          poids_urgence=0.15):
    """
    Trouve le meilleur chauffeur + position pour le nouveau colis.
    nouveau_colis doit contenir les mêmes clés que les colis de dfc_propre :
        id, lat, lon, poids_kg, date_creation, description,
        frais_tnd, priorite, mode_paiement, type_client, contenu
    La deadline_sec est calculée automatiquement depuis date_creation.
    """
    # Calcul automatique deadline depuis Date de création
    nouveau_colis['deadline_sec'] = _deadline_sec(nouveau_colis.get('date_creation'))

    candidats, logs = [], []

    for ch in flotte:
        stops_restants = ch['tournee'][ch['prochain_stop_idx']:]
        for pos in range(len(stops_restants) + 1):
            res = evaluer_insertion(ch, nouveau_colis, pos)
            logs.append({'chauffeur': ch['nom'], 'position': pos,
                         'faisable': res['ok'], 'delta_km': res.get('delta_km', '—'),
                         'motif': res.get('motif', '')})
            if res['ok']:
                candidats.append({'ch': ch, 'pos': pos, **res})

    if not candidats:
        print('\n❌ AUCUNE INSERTION POSSIBLE')
        print('   → Colis replanifié sur la tournée du lendemain (24h non dépassées)')
        return None, logs

    # Score composite
    max_km = max(c['delta_km']        for c in candidats) or 1
    max_dt = max(c['temps_total_sec'] for c in candidats) or 1

    def score(c):
        s_carb = 1 - c['delta_km'] / max_km
        s_tps  = 1 - c['temps_total_sec'] / max_dt
        marge  = nouveau_colis['deadline_sec'] - c['eta_nouveau_sec']
        s_urg  = min(1.0, marge / max(nouveau_colis['deadline_sec'], 1))
        return poids_carburant * s_carb + poids_temps * s_tps + poids_urgence * s_urg

    meilleur = max(candidats, key=score)

    # Appliquer l'insertion dans la flotte
    meilleur['ch']['tournee']          = meilleur['tournee_finale']
    meilleur['ch']['poids_charge_kg'] += nouveau_colis['poids_kg']
    meilleur['ch']['distance_km']      = round(meilleur['ch']['distance_km'] + meilleur['delta_km'], 2)

    # Affichage
    gain_net = nouveau_colis.get('frais_tnd', TARIF_LIVRAISON_TND) - meilleur['delta_cout_tnd']
    marge_dl = (nouveau_colis['deadline_sec'] - meilleur['eta_nouveau_sec']) / 3600
    SEP = '─' * 70
    print(f'\n{"═"*70}')
    print(f'  📦 INSERTION DYNAMIQUE — {nouveau_colis["id"]}')
    print(f'{"═"*70}')
    print(f'  Chauffeur sélectionné  : {meilleur["ch"]["nom"]}')
    print(f'  Position dans file     : stop #{meilleur["pos"] + 1} / {len(meilleur["ch"]["tournee"])} stops total')
    print(f'  {SEP}')
    print(f'  Détour supplémentaire  : +{meilleur["delta_km"]:.3f} km')
    print(f'  Surcoût carburant      : +{meilleur["delta_cout_tnd"]:.3f} TND')
    print(f'  Tarif encaissé         : +{nouveau_colis.get("frais_tnd", TARIF_LIVRAISON_TND):.2f} TND')
    print(f'  Gain net sur ce colis  :  {gain_net:+.3f} TND  {"✅ rentable" if gain_net > 0 else "🔴 déficitaire"}')
    print(f'  {SEP}')
    print(f'  Date de création       : {nouveau_colis.get("date_creation", "—")}')
    print(f'  Deadline (créa + 24h)  : {nouveau_colis["deadline_sec"]/3600:.2f}h depuis départ entrepôt')
    print(f'  ETA livraison estimée  : {meilleur["eta_nouveau_sec"]/3600:.2f}h depuis départ entrepôt')
    print(f'  Marge avant expiration : {marge_dl:.2f}h')
    print(f'{"═"*70}')

    print(f'\n📍 NOUVEAU PLANNING — {meilleur["ch"]["nom"]} :')
    for i, c in enumerate(meilleur['ch']['tournee']):
        if i < meilleur['ch']['prochain_stop_idx']:
            marker = '  ✅'
        elif c['id'] == nouveau_colis['id']:
            marker = '  🆕'
        else:
            marker = '  ➡️ '
        print(f"{marker} [{i+1}] {c['id']} — {c['description']} ({c['priorite']})")

    stops_r = meilleur['ch']['tournee'][meilleur['ch']['prochain_stop_idx']:]
    coords  = [f"{ENTREPOT_COORD[0]},{ENTREPOT_COORD[1]}"]
    coords += [f"{s['lat']},{s['lon']}" for s in stops_r]
    coords += [f"{ENTREPOT_COORD[0]},{ENTREPOT_COORD[1]}"]
    print(f'\n🗺️  GPS : https://www.google.com/maps/dir/' + '/'.join(coords))

    return meilleur, logs


# ─────────────────────────────────────────────────────────────────────────────
# ▶▶▶  EXÉCUTIOn
# ─────────────────────────────────────────────────────────────────────────────

# 1. Conversion OR-Tools → flotte native (lit dfc_propre + flotte_optimale)
flotte = construire_flotte_depuis_ortools(dfc_propre, flotte_optimale)

print(f'✅ Flotte reconstruite : {len(flotte)} chauffeur(s)\n')
for ch in flotte:
    print(f'  🚛 {ch["nom"]} | {len(ch["tournee"])} colis | '
          f'{ch["distance_km"]:.1f} km | {ch["poids_charge_kg"]:.1f} kg chargés')

# ─────────────────────────────────────────────────────────────────────────────
# 2. Rentabilité — tournée initiale du matin
# ─────────────────────────────────────────────────────────────────────────────
print('\n▶ BILAN RENTABILITÉ — TOURNÉE INITIALE :')
afficher_rentabilite(flotte)



plan dynamique 

#calcul de rentabilite  apres insertion

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Nouveau colis arrivé pendant la journée
#    → Remplir ci-dessous avec les vraies valeurs reçues de la plateforme
# ─────────────────────────────────────────────────────────────────────────────
nouveau_colis = {
    'id'           : 'C_NEW_001',
    'lat'          : 36.845,
    'lon'          : 10.215,
    'poids_kg'     : 2.0,
    'date_creation': datetime.now().isoformat(),   # ← timestamp réel de la commande
    'description'  : 'El Menzah 6',
    'frais_tnd'    : 8.0,                          # ← Frais_Livraison_TND de la commande
    'priorite'     : 'Express',
    'mode_paiement': 'Paiement à la livraison',
    'type_client'  : 'B2C (Particulier)',
    'contenu'      : 'Vêtements',
}

meilleur, logs = inserer_nouveau_colis(flotte, nouveau_colis)

# ─────────────────────────────────────────────────────────────────────────────
# 4. Rentabilité mise à jour après insertion
# ─────────────────────────────────────────────────────────────────────────────
if meilleur:
    print('\n▶ BILAN RENTABILITÉ — APRÈS INSERTION DU NOUVEAU COLIS :')
    afficher_rentabilite(flotte)

forecasting 
